In [ ]:
# Durian leaf disease classifier training notebook for Google Colab.
# Runtime suggestion: Runtime > Change runtime type > GPU.
# Every cell is runnable Python; comments explain the workflow without Markdown cells.

!pip -q install -U tf2onnx onnx onnxruntime roboflow

import json
import os
import random
import re
import shutil
import unicodedata
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.metrics import confusion_matrix

SEED = 1337
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 5
# Class names are the dataset's actual folder names (DLD_FinalDataset), in a FIXED order.
# This order maps directly to the label indices used by the model, server, and frontend.
CLASS_NAMES = [
    "HEALTHY_LEAF",
    "ALGAL_LEAF_SPOT",
    "LEAF_BLIGHT",
    "PHOMOPSIS_LEAF_SPOT",
    "ALLOCARIDARA_ATTACK",
]

tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

WORK_DIR = Path("/content/durian_leaf_classifier")
RAW_DIR = WORK_DIR / "raw_dataset"
NORMALIZED_DIR = WORK_DIR / "normalized_dataset"
BEST_KERAS_MODEL_PATH = WORK_DIR / "best_leaf_disease_model.keras"
ONNX_MODEL_PATH = Path("/content/leaf_disease_model.onnx")
LABELS_PATH = Path("/content/leaf_labels.json")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Dataset source selection.
# Path A: fill all Roboflow variables below, then run the cell.
# Path B: leave any Roboflow variable blank and upload a .zip containing class folders.
# No API key is hardcoded in this notebook.

ROBOFLOW_API_KEY = ""
ROBOFLOW_WORKSPACE = ""
ROBOFLOW_PROJECT = ""
ROBOFLOW_VERSION = ""

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

use_roboflow = all([
    ROBOFLOW_API_KEY.strip(),
    ROBOFLOW_WORKSPACE.strip(),
    ROBOFLOW_PROJECT.strip(),
    str(ROBOFLOW_VERSION).strip(),
])

if use_roboflow:
    from roboflow import Roboflow

    print("Downloading Roboflow classification dataset as folder format...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(int(ROBOFLOW_VERSION)).download("folder", location=str(RAW_DIR))
    RAW_DATASET_DIR = Path(dataset.location)
else:
    from google.colab import files

    print("Upload one .zip file containing image folders for the 4 classes.")
    uploaded = files.upload()
    zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
    if len(zip_files) != 1:
        raise ValueError(f"Expected exactly one .zip upload, got {zip_files}")
    zip_path = Path("/content") / zip_files[0]
    shutil.unpack_archive(str(zip_path), str(RAW_DIR))
    RAW_DATASET_DIR = RAW_DIR

print("Raw dataset directory:", RAW_DATASET_DIR)


In [ ]:
# This dataset is already split into train/val/test with one folder per class
# (e.g. DLD_FinalDataset_224_spit/{train,val,test}/{HEALTHY_LEAF, ...}).
# No alias normalization is needed — locate the split root and verify the 5 class folders.

def find_split_root(base_dir):
    base_dir = Path(base_dir)
    candidates = [base_dir, *sorted(p for p in base_dir.rglob("*") if p.is_dir())]
    for path in candidates:
        child_dirs = {c.name.lower() for c in path.iterdir() if c.is_dir()}
        if "train" in child_dirs and ({"val", "valid", "test"} & child_dirs):
            return path
    raise ValueError(f"Could not find a folder containing train/ and val/test under {base_dir}")

DATASET_ROOT = find_split_root(RAW_DATASET_DIR)
TRAIN_DIR = DATASET_ROOT / "train"
if (DATASET_ROOT / "val").exists():
    VAL_DIR = DATASET_ROOT / "val"
elif (DATASET_ROOT / "valid").exists():
    VAL_DIR = DATASET_ROOT / "valid"
else:
    VAL_DIR = DATASET_ROOT / "test"

print("Dataset root:", DATASET_ROOT)
print("Train dir:", TRAIN_DIR)
print("Val dir:  ", VAL_DIR)

for split_dir in [TRAIN_DIR, VAL_DIR]:
    print(f"\n{split_dir.name}:")
    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        n = len([p for p in class_dir.glob("*") if p.is_file()]) if class_dir.exists() else 0
        if n == 0:
            raise ValueError(f"Missing or empty class folder: {class_dir}")
        print(f"  {class_name}: {n} images")

In [ ]:
# Build RGB 224x224 datasets directly from the existing train/ and val/ folders.
# Browser-compatible preprocessing is exactly float32 RGB divided by 255.
# No ImageNet mean/std normalization is used.

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False,
)

validation_image_paths = list(getattr(val_ds_raw, "file_paths", []))

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="light_augmentation")

def preprocess_train(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    images = data_augmentation(images, training=True)
    return images, labels

def preprocess_eval(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds_raw.map(preprocess_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print("Class order from dataset:", train_ds_raw.class_names)
print("Validation images available for sanity test:", len(validation_image_paths))

In [ ]:
# Model: MobileNetV3Small backbone, GlobalAveragePooling2D, Dropout, Dense(NUM_CLASSES, softmax).
# include_preprocessing=False so the model expects [0,1] browser-style inputs.
# Improvements over the first ~78% run: class weights (help the hard / smaller classes
# like Algal & Phomopsis), a deeper fine-tune (unfreeze the last 60 layers), and a
# slightly higher fine-tune learning rate (3e-5) to push past the plateau.

def make_backbone():
    kwargs = dict(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
    try:
        return tf.keras.applications.MobileNetV3Small(**kwargs, include_preprocessing=False)
    except TypeError:
        return tf.keras.applications.MobileNetV3Small(**kwargs)

base_model = make_backbone()
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3), name="image")
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(0.25, name="dropout")(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
model = tf.keras.Model(inputs, outputs, name="durian_leaf_disease_classifier")

# Class weights from the train split: penalize mistakes on smaller/harder classes more,
# which reduces the "predict everything as healthy" bias seen in the first run.
train_counts = [len([p for p in (TRAIN_DIR / c).glob("*") if p.is_file()]) for c in CLASS_NAMES]
total_train = sum(train_counts)
class_weight = {i: total_train / (NUM_CLASSES * cnt) for i, cnt in enumerate(train_counts)}
print("Train counts:", dict(zip(CLASS_NAMES, train_counts)))
print("Class weights:", class_weight)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    BEST_KERAS_MODEL_PATH,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1,
)
early_stop_frozen = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=[checkpoint, early_stop_frozen],
)

# Fine-tune a larger portion of the backbone (last 60 layers) at a low learning rate.
base_model.trainable = True
fine_tune_at = max(0, len(base_model.layers) - 60)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

early_stop_finetune = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weight,
    callbacks=[checkpoint, early_stop_finetune],
)

# Reload the best checkpoint across both training phases before evaluation/export.
best_model = tf.keras.models.load_model(BEST_KERAS_MODEL_PATH)
val_loss, val_accuracy = best_model.evaluate(val_ds, verbose=0)
print(f"Validation/Test holdout accuracy: {val_accuracy:.4f}")
print(f"Validation/Test holdout loss: {val_loss:.4f}")
if val_accuracy < 0.85:
    print("WARNING: accuracy is below 0.85. Add more balanced images, train for more epochs, or tune augmentation. Exporting the best early-stopped model anyway.")

In [ ]:
# Print confusion matrix and per-class accuracy in the fixed index order.

y_true = []
y_pred = []

for images, labels in val_ds:
    probabilities = best_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(probabilities, axis=1).tolist())

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
print("Confusion matrix rows=true, cols=pred, order=0..3:")
print(cm)

print("Per-class accuracy:")
for index, class_name in enumerate(CLASS_NAMES):
    total = int(cm[index].sum())
    correct = int(cm[index, index])
    class_accuracy = correct / total if total else 0.0
    print(f"{index}={class_name}: {class_accuracy:.4f} ({correct}/{total})")


In [ ]:
# Export ONNX with NHWC float32 input [1,224,224,3] and softmax output [1,5].
# Opset 15 matches tf2onnx's modern default/tested range while preserving NHWC input.
# Write leaf_labels.json for the 5 dataset classes in the fixed index order.

import tf2onnx

input_signature = (tf.TensorSpec([1, 224, 224, 3], tf.float32, name="input"),)
onnx_model, _ = tf2onnx.convert.from_keras(
    best_model,
    input_signature=input_signature,
    opset=15,
    output_path=str(ONNX_MODEL_PATH),
)

LABELS_JSON_TEXT = '{"0":{"vi":"Lá khỏe","en":"Healthy"},"1":{"vi":"Đốm rong","en":"Algal Leaf Spot"},"2":{"vi":"Cháy lá","en":"Leaf Blight"},"3":{"vi":"Đốm lá Phomopsis","en":"Phomopsis Leaf Spot"},"4":{"vi":"Rầy nhảy gây hại","en":"Allocaridara (Psyllid) Attack"}}'
LABELS_PATH.write_text(LABELS_JSON_TEXT, encoding="utf-8")
leaf_labels = json.loads(LABELS_JSON_TEXT)

print("Wrote:", ONNX_MODEL_PATH)
print("Wrote:", LABELS_PATH)
print("Labels JSON:", LABELS_PATH.read_text(encoding="utf-8"))

In [ ]:
# Final ONNX sanity test: load leaf_disease_model.onnx, pick one validation image,
# preprocess exactly like browser inference (RGB, resize 224x224, float32 / 255),
# run inference, print predicted class/name/probabilities, then download artifacts.

import onnxruntime as ort
from google.colab import files

if not validation_image_paths:
    raise ValueError("No validation image paths found for ONNX sanity test.")

sanity_image_path = Path(validation_image_paths[0])
image = Image.open(sanity_image_path).convert("RGB").resize(IMG_SIZE)
onnx_input = np.asarray(image, dtype=np.float32) / 255.0
onnx_input = np.expand_dims(onnx_input, axis=0)

session = ort.InferenceSession(str(ONNX_MODEL_PATH), providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
probabilities = session.run([output_name], {input_name: onnx_input})[0][0]

if probabilities.shape[0] != NUM_CLASSES:
    raise ValueError(f"Expected ONNX output with 4 probabilities, got shape {probabilities.shape}")

predicted_index = int(np.argmax(probabilities))
predicted_label = leaf_labels[str(predicted_index)]

print("Sanity image:", sanity_image_path)
print("ONNX input name:", input_name, "shape:", onnx_input.shape, "dtype:", onnx_input.dtype)
print("ONNX output name:", output_name, "shape:", probabilities.reshape(1, -1).shape)
print("Predicted class index:", predicted_index)
print("Predicted class name:", predicted_label["en"], "/", predicted_label["vi"])
print("Probabilities in fixed order:")
for index, class_name in enumerate(CLASS_NAMES):
    print(f"{index}={class_name}: {float(probabilities[index]):.6f}")

files.download(str(ONNX_MODEL_PATH))
files.download(str(LABELS_PATH))
